# Simple AWG Example

Generate simple `int16` DAC waveforms, preview them, load them into the DAC BRAM players, and enable or disable RF output explicitly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware import signals

In [ ]:
ol = OverlayController()
info = ol.info()
info

In [ ]:
DAC0_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
DAC2_SR = float(info["rfdc"]["dac2_sampling_rate_gsps"]) * 1e9
DAC0_LEN = int(info["dac0"]["bram_int16_samples"])
DAC2_LEN = int(info["dac2"]["bram_int16_samples"])
DAC_PEAK = int(0.8 * np.iinfo(np.int16).max)

DAC0_SR, DAC2_SR, DAC0_LEN, DAC2_LEN, DAC_PEAK

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def plot_waveform(waveform, sample_rate, samples=2048, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax


def plot_spectrum(waveform, sample_rate, title="Spectrum"):
    data = np.asarray(waveform, dtype=float)
    spectrum = np.fft.rfft(data)
    freqs_mhz = np.fft.rfftfreq(data.size, d=1 / sample_rate) / 1e6
    magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freqs_mhz, magnitude_db)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Waveforms

The signal helpers return floating-point amplitudes. Here the amplitude is already in DAC-code units, then clipped and rounded to `int16`.

In [ ]:
dac0_waveform = to_int16(
    signals.sine(freq_hz=100e6, sample_rate=DAC0_SR, num_samples=DAC0_LEN, amplitude=DAC_PEAK)
)

dac2_waveform = to_int16(
    signals.sawtooth(freq_hz=50e6, sample_rate=DAC2_SR, num_samples=DAC2_LEN, amplitude=DAC_PEAK)
)

dac0_waveform.dtype, dac0_waveform.shape, dac2_waveform.dtype, dac2_waveform.shape

In [ ]:
plot_waveform(dac0_waveform, DAC0_SR, title="DAC0 waveform")
plot_spectrum(dac0_waveform, DAC0_SR, title="DAC0 spectrum")

plot_waveform(dac2_waveform, DAC2_SR, title="DAC2 waveform")
plot_spectrum(dac2_waveform, DAC2_SR, title="DAC2 spectrum");

## Program DAC Players

In [ ]:
ol.dac0.load_waveform(dac0_waveform)
ol.dac2.load_waveform(dac2_waveform)

ol.info()

## Enable Outputs

Only run this cell when the RF chain and instruments are ready.

In [ ]:
ol.dac0.enable()
ol.dac2.enable()

ol.dac0.is_enabled(), ol.dac2.is_enabled()

## Disable Outputs

Run this before changing cabling or loading a different waveform.

In [ ]:
ol.dac0.disable()
ol.dac2.disable()

ol.info()